# 02 — Phylogenetic Signal Analysis

This notebook:
- Visualises the phylogenetic tree alongside metadata
- Quantifies phylogenetic signal in each target label (Blomberg's K)
- Visualises clade holdout CV fold assignments
- Compares clade-holdout vs random CV accuracy estimates
- Computes and visualises phylogenetic eigenvectors

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

from fungal_classifier.utils.io import load_metadata, load_feature_blocks
from fungal_classifier.evaluation.phylo_cv import (
    load_tree,
    get_patristic_distances,
    phylogenetic_eigenvectors,
    assign_clades_from_taxonomy,
    assign_clades_from_tree,
    CladeHoldoutCV,
    blombergs_k,
)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120})
sns.set_style("whitegrid")


## 1. Load data

In [ ]:
METADATA_PATH = Path("../data/raw/metadata.tsv")
TREE_PATH     = Path("../data/raw/phylogeny.nwk")

metadata = load_metadata(METADATA_PATH)
tree = load_tree(str(TREE_PATH))
genome_ids = metadata.index.tolist()
print(f"Genomes: {len(genome_ids)}")


## 2. Tree topology — coloured by taxonomy order

Using ete3 for tree visualisation. If ete3 is not available, a distance-matrix heatmap is shown instead.

In [ ]:
try:
    from ete3 import Tree, TreeStyle, NodeStyle, faces, AttrFace
    import colorsys

    # Build colour map for orders
    orders = metadata["taxonomy_order"].dropna().unique()
    hues = np.linspace(0, 1, len(orders), endpoint=False)
    order_colors = {o: "#{:02x}{:02x}{:02x}".format(
        *[int(x*255) for x in colorsys.hsv_to_rgb(h, 0.7, 0.85)])
        for o, h in zip(sorted(orders), hues)}

    for leaf in tree.iter_leaves():
        gid = leaf.name
        if gid in metadata.index:
            order = metadata.loc[gid, "taxonomy_order"]
            ns = NodeStyle()
            ns["bgcolor"] = order_colors.get(order, "#cccccc")
            ns["size"] = 0
            leaf.set_style(ns)

    ts = TreeStyle()
    ts.show_leaf_name = False
    ts.mode = "c"    # circular layout
    ts.arc_start = -180
    ts.arc_span  = 360
    print("Tree loaded. Call tree.render() or tree.show() to visualise interactively.")
    print(f"Tips: {len(tree.get_leaves())}  |  Internal nodes: {sum(1 for n in tree.traverse() if not n.is_leaf())}")

except ImportError:
    print("ete3 not available — showing distance matrix heatmap instead.")
    # Subsample for speed
    sample_ids = genome_ids[:200] if len(genome_ids) > 200 else genome_ids
    D = get_patristic_distances(tree, sample_ids)
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(D.values, ax=ax, cmap="viridis", xticklabels=False, yticklabels=False)
    ax.set_title("Patristic distance matrix (first 200 genomes)")
    plt.tight_layout()
    plt.show()


## 3. Blomberg's K — phylogenetic signal per target label

In [ ]:
# Compute patristic distance matrix (subsample if large)
SAMPLE_N = min(1000, len(genome_ids))
sample_ids = pd.Series(genome_ids).sample(SAMPLE_N, random_state=42).tolist()
print(f"Computing patristic distances for {SAMPLE_N} genomes...")
D = get_patristic_distances(tree, sample_ids)
print("Done.")


In [ ]:
from sklearn.preprocessing import LabelEncoder

targets = ["taxonomy_order", "taxonomy_class", "ecological_niche", "lifestyle"]
targets = [t for t in targets if t in metadata.columns]

K_results = []
for target in targets:
    y = metadata.loc[sample_ids, target].dropna()
    common = y.index.intersection(D.index)
    if len(common) < 10:
        continue
    le = LabelEncoder()
    y_enc = pd.Series(le.fit_transform(y.loc[common]), index=common)
    K = blombergs_k(y_enc, D.loc[common, common])
    K_results.append({"target": target, "K": K, "n": len(common)})
    print(f"  {target:30s}  K = {K:.3f}  (n={len(common)})")

K_df = pd.DataFrame(K_results)
fig, ax = plt.subplots(figsize=(7, 3.5))
colors = ["#d62728" if K > 1 else "#1a6fa8" if K > 0.5 else "#aec7e8"
          for K in K_df["K"]]
bars = ax.barh(K_df["target"], K_df["K"], color=colors, edgecolor="white")
ax.axvline(1.0, color="black", linestyle="--", linewidth=1, label="K=1 (Brownian motion)")
ax.axvline(0.0, color="grey", linestyle=":", linewidth=1)
ax.set_xlabel("Blomberg's K")
ax.set_title("Phylogenetic signal per target label")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


## 4. Clade holdout CV — fold assignment visualisation

In [ ]:
CLADE_LEVEL = "order"
N_FOLDS     = 10

clade_labels = assign_clades_from_taxonomy(metadata, clade_level=CLADE_LEVEL)
cv = CladeHoldoutCV(clade_labels=clade_labels, n_folds=N_FOLDS, random_seed=42)
fold_summary = cv.fold_summary()

print(f"Clade holdout CV — {N_FOLDS} folds at {CLADE_LEVEL} level")
print(fold_summary.groupby("fold")["n_genomes"].sum().rename("genomes_in_test"))


In [ ]:
# Visualise fold balance
fold_counts = fold_summary.groupby("fold")["n_genomes"].sum().reset_index()
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(fold_counts["fold"], fold_counts["n_genomes"], color="#1a6fa8", edgecolor="white")
ax.axhline(fold_counts["n_genomes"].mean(), color="red", linestyle="--",
           label=f"Mean: {fold_counts['n_genomes'].mean():.0f}")
ax.set_xlabel("Fold")
ax.set_ylabel("# Genomes in test set")
ax.set_title(f"Clade holdout CV — test set sizes ({CLADE_LEVEL}-level holdout)")
ax.legend()
plt.tight_layout()
plt.show()


## 5. CV strategy comparison — clade holdout vs random

Expected result: random CV overestimates accuracy for taxonomy; clade holdout gives a more realistic estimate.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from fungal_classifier.utils.io import load_feature_blocks

blocks = load_feature_blocks(Path("../data/features/"))
if blocks:
    # Use domain block as a quick proxy
    block_name = "domains" if "domains" in blocks else list(blocks.keys())[0]
    X_block = blocks[block_name]
    TARGET = "taxonomy_order"
    y_block = metadata.loc[X_block.index.intersection(metadata.index), TARGET].dropna()
    X_block = X_block.loc[y_block.index].fillna(0)

    clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)

    # Random stratified CV
    random_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    random_scores = cross_val_score(clf, X_block, y_block, cv=random_cv,
                                    scoring="f1_macro", n_jobs=-1)

    # Clade holdout CV
    clade_labels_block = clade_labels.loc[y_block.index]
    clade_cv = CladeHoldoutCV(clade_labels=clade_labels_block, n_folds=5)
    clade_scores = cross_val_score(clf, X_block, y_block, cv=clade_cv,
                                   scoring="f1_macro", n_jobs=-1)

    print(f"Block: {block_name} | Target: {TARGET}")
    print(f"  Random CV       F1-macro: {random_scores.mean():.3f} ± {random_scores.std():.3f}")
    print(f"  Clade holdout   F1-macro: {clade_scores.mean():.3f} ± {clade_scores.std():.3f}")
    print(f"  Inflation:      {random_scores.mean() - clade_scores.mean():.3f}")

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.boxplot([random_scores, clade_scores],
               labels=["Random CV", "Clade holdout CV"],
               patch_artist=True,
               boxprops=dict(facecolor="#aec7e8"),
               medianprops=dict(color="black", linewidth=2))
    ax.set_ylabel("F1-macro")
    ax.set_title(f"CV strategy comparison — {block_name} block, {TARGET}")
    plt.tight_layout()
    plt.show()
else:
    print("No feature blocks found. Run scripts/01_build_features.py first.")


## 6. Phylogenetic eigenvectors — PCoA of patristic distances

In [ ]:
print("Computing phylogenetic eigenvectors (PCoA on patristic distances)...")
phylo_eig = phylogenetic_eigenvectors(D, n_components=10)
print(f"Phylo eigenvectors shape: {phylo_eig.shape}")

# Plot PC1 vs PC2 coloured by taxonomy
common = phylo_eig.index.intersection(metadata.index)
orders = metadata.loc[common, "taxonomy_order"].fillna("Unknown")
unique_orders = sorted(orders.unique())
palette = sns.color_palette("tab20", n_colors=len(unique_orders))
c_map = dict(zip(unique_orders, palette))
colors = [c_map[o] for o in orders]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (x_col, y_col) in zip(axes, [("phylo_pc1", "phylo_pc2"), ("phylo_pc3", "phylo_pc4")]):
    ax.scatter(phylo_eig.loc[common, x_col], phylo_eig.loc[common, y_col],
               c=colors, s=10, alpha=0.7)
    ax.set_xlabel(x_col); ax.set_ylabel(y_col)
    ax.set_title(f"Phylogenetic eigenvectors: {x_col} vs {y_col}")

handles = [mpatches.Patch(color=c, label=o) for o, c in c_map.items()]
axes[1].legend(handles=handles, bbox_to_anchor=(1.02, 1), loc="upper left",
               fontsize=7, title="Taxonomy order")
plt.tight_layout()
plt.show()
